In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import RYGate
from qiskit.circuit import Gate

# First, run your butterfly on random data to get fresh angles
import random
import math

# Your butterfly function here...
def run_butterfly_on_random():
    z = np.random.randn(16)  # 16 random values
    result = butterfly(z)  # Your butterfly function
    
    # Extract angles into stage-based dictionary
    angles_dict = {}
    for i, stage in enumerate(result['stages'], 1):
        angles_dict[f'stage{i}'] = [pair['angle'] for pair in stage]
    
    return angles_dict

# Get fresh angles
angles = run_butterfly_on_random()

print("Butterfly angles by stage:")
for stage, stage_angles in angles.items():
    print(f"{stage}: {[f'{a:.2f}°' for a in stage_angles]}")

# Function to convert degrees to radians and double for RY gate
def prep_angle(deg):
    """Convert degrees to radians and double for RY gate"""
    return 2 * np.radians(deg)

# Function to get angle by stage and index
def get_angle(stage, index):
    """Get angle by stage and index, ready for RY gate"""
    stage_key = f'stage{stage}'
    deg = angles[stage_key][index]
    return prep_angle(deg)

# ============================================================
#  CREATE A SINGLE CUSTOM GATE WITH ALL YOUR ROTATIONS
# ============================================================

# First, create a sub-circuit that contains all your gates
sub_qc = QuantumCircuit(4, name='Butterfly_Rot')

# Your specific mapping pattern:
theta1 = get_angle(1, 0)
theta2 = get_angle(1, 1)
theta3 = get_angle(1, 2)
theta4 = get_angle(1, 3)
theta5 = get_angle(1, 4)
theta6 = get_angle(1, 5)
theta7 = get_angle(1, 6)
theta8 = get_angle(1, 7)
theta9 = get_angle(2, 0)
theta10 = get_angle(2, 1)
theta11 = get_angle(2, 2)
theta12 = get_angle(2, 3)
theta13 = get_angle(3, 0)
theta14 = get_angle(3, 1)
theta15 = get_angle(4, 0)

print("\nAdding gates to composite circuit...")

# Add all your gates to the sub-circuit
# Gate 1
gate = RYGate(theta1) #.control(num_ctrl_qubits=0, ctrl_state='000')
sub_qc.append(gate, [0, 1, 2, 3])

# Gate 2
gate = RYGate(theta2).control(num_ctrl_qubits=3, ctrl_state='100')
sub_qc.append(gate, [0, 1, 2, 3])

# Gate 3
gate = RYGate(theta9).control(num_ctrl_qubits=3, ctrl_state='000')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 4
gate = RYGate(theta3).control(num_ctrl_qubits=3, ctrl_state='010')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 5
gate = RYGate(theta4).control(num_ctrl_qubits=3, ctrl_state='110')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 6
gate = RYGate(theta10).control(num_ctrl_qubits=3, ctrl_state='010')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 7
gate = RYGate(theta13).control(num_ctrl_qubits=3, ctrl_state='001')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 8
gate = RYGate(theta5).control(num_ctrl_qubits=3, ctrl_state='001')
sub_qc.append(gate, [0, 1, 3, 2])

# Gate 9
gate = RYGate(theta6).control(num_ctrl_qubits=3, ctrl_state='101')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 10
gate = RYGate(theta11).control(num_ctrl_qubits=3, ctrl_state='001')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 11
gate = RYGate(theta7).control(num_ctrl_qubits=3, ctrl_state='011')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 12
gate = RYGate(theta8).control(num_ctrl_qubits=3, ctrl_state='111')
sub_qc.append(gate, [0, 1, 2,3])

# Gate 13
gate = RYGate(theta12).control(num_ctrl_qubits=3, ctrl_state='011')
sub_qc.append(gate, [0, 1, 2, 3])

# Gate 14
gate = RYGate(theta14).control(num_ctrl_qubits=3, ctrl_state='001')
sub_qc.append(gate, [0,1,2, 3])

# Gate 15
gate = RYGate(theta15).control(num_ctrl_qubits=3, ctrl_state='000')
sub_qc.append(gate, [0,1,2, 3])

# Convert the sub-circuit to a custom gate
butterfly_gate = sub_qc.to_gate(label='Butterfly_Rot')
butterfly_gate.name = "Butterfly_Rot"

# ============================================================
#  USE YOUR SINGLE CUSTOM GATE
# ============================================================

# Create your main circuit
qc = QuantumCircuit(4)

# Add your single custom gate (contains all 15 rotations)
qc.append(butterfly_gate, [0, 1, 2, 3])

print("\n" + "="*60)
print("MAIN CIRCUIT WITH SINGLE BUTTERFLY GATE:")
print("="*60)
print(qc.draw())

# If you want to see what's inside your custom gate:
print("\n" + "="*60)
print("INSIDE THE BUTTERFLY GATE (all 15 rotations):")
print("="*60)
print(sub_qc.draw())

# Optional: You can also create a controlled version of your custom gate
# This adds another layer of control on top of all your rotations
butterfly_gate_controlled = butterfly_gate.control(num_ctrl_qubits=1)
qc_controlled = QuantumCircuit(5)
qc_controlled.append(butterfly_gate_controlled, [0, 1, 2, 3, 4])  # Q0 control, Q1-4 are the original qubits

print("\n" + "="*60)
print("CIRCUIT WITH CONTROLLED BUTTERFLY GATE:")
print("="*60)
print(qc_controlled.draw())

# You can also create different versions with different parameters
def create_butterfly_gate(angles_dict):
    """Factory function to create butterfly gate with given angles"""
    def get_angle(stage, index):
        stage_key = f'stage{stage}'
        deg = angles_dict[stage_key][index]
        return 2 * np.radians(deg)
    
    sub_qc = QuantumCircuit(4, name='Butterfly_Rot')
    
    # Add all gates (same as above but with dynamic angles)
    theta1 = get_angle(1, 0)
    sub_qc.append(RYGate(theta1).control(3, '000'), [0, 1, 2, 3])
    
    theta2 = get_angle(1, 1)
    sub_qc.append(RYGate(theta2).control(3, '100'), [0, 1, 2, 3])
    
    # ... add all other gates
    
    return sub_qc.to_instruction()

# Create a new gate with different angles
new_angles = run_butterfly_on_random()
new_butterfly_gate = create_butterfly_gate(new_angles)

qc2 = QuantumCircuit(4)
qc2.append(new_butterfly_gate, [0, 1, 2, 3])

print("\n" + "="*60)
print("ANOTHER BUTTERFLY GATE (different angles):")
print("="*60)
print(qc2.draw())

NameError: name 'butterfly' is not defined